# 🎯 [C204_M09] AAR 기반 무전 위조 오판정 음성 분석 및 음향 특징 추출 파라미터 재보정
### Google Colab 실습 환경 (지휘통제(C2) / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/C204/c204_m09_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 지휘통제(C2) (Level 3) |
| **미션 ID** | C204_M09 |
| **미션 주제** | AAR 기반 무전 위조 오판정 음성 분석 및 음향 특징 추출 파라미터 재보정 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 지휘관 명의의 위조 음성·영상과 무전 기만이 실제로 보고되면서, 생성형 AI 시대에 '들리는 목소리가 정말 그 지휘관인가'를 확인하는 일이 지휘망 보안의 새로운 과제가 되었다. 본 프로젝트에서는 실제 음성 대신 탐지 학습용 합성 신호(배음 신호에 진짜 음성의 자연스러운 흔들림과 문헌상 알려진 위조 흔적을 통계적으로 표시한 추상 신호)를 코드로 만들어, 데이터셋 규격 설계, 무전 채널 잡음 모델링과 MFCC·스펙트로그램 전처리, 규칙 기반 Baseline, 신경망 판별 엔진, 진위 신뢰도 점수, C2 보안 상황판, 통신보안관 2차 검증(HITL), 전자전 재밍 스트레스 시험, AAR 재보정, 최종 통합까지 위조 음성 탐지 파이프라인을 단계별로 구현한다. 음성을 생성하거나 변조하는 공격 기법은 다루지 않으며, 오직 수신 측의 탐지·검증 기능만 구현한다. 최종 목표는 잡음이 심한 무전 환경에서도 위조 명령을 놓치지 않고 오경보를 줄이며, 모든 긴급 명령의 진위 확정이 사람의 인증·청음 확인을 거치도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/c204_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_c204.py` | 전처리, Baseline/신경망 모델, 신뢰도 모듈, 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/c204_data_spec.md`, `docs/c204_hitl_guide.md`, `docs/c204_aar_report.md`, `docs/c204_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 기술/지휘관 문서 |
| **`configs/`**| `configs/c204_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/c204_stress_test.py` | 전자전 재밍/잡음 스트레스 시험 및 단위 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/c204_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [1]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


# ----------------------------------------------------------------------------
# [군 표준 산출물 디렉토리 편철 준비 및 가상 데이터셋 무결성 확인]
# ----------------------------------------------------------------------------
output_dirs = [
    'data/schema',
    'docs',
    'src/preprocess',
    'src/models',
    'src/verification',
    'src/ui',
    'src/hitl',
    'tests',
    'configs'
]

print("=== [1] 군 표준 산출물 보관용 폴더 생성 ===")
for d in output_dirs:
    os.makedirs(d, exist_ok=True)
    print(f"  📁 디렉토리 준비 완료: {d}/")

print("\n=== [2] 가상 데이터셋 무결성 점검 ===")
data_dir = 'uploads/C204'
if not os.path.exists(data_dir):
    data_dir = '/home/subokim/Workspace/milicontent2/uploads/C204'

dataset_count = 0
for i in range(1, 11):
    fname = os.path.join(data_dir, f'c204_m{i:02d}_dataset.json')
    if os.path.exists(fname):
        with open(fname, 'r', encoding='utf-8') as f:
            d = json.load(f)
        print(f"  ✅ 데이터셋 확인: c204_m{i:02d}_dataset.json ({d.get('description', '')[:32]}...)")
        dataset_count += 1
    else:
        print(f"  ⚠️ 데이터셋 확인 필요: {fname}")

print(f"\n총 {dataset_count}/10개 가상 데이터셋 무결성 검증 완료. 훈련 준비 완료!\n")


=== [1] 군 표준 산출물 보관용 폴더 생성 ===
  📁 디렉토리 준비 완료: data/schema/
  📁 디렉토리 준비 완료: docs/
  📁 디렉토리 준비 완료: src/preprocess/
  📁 디렉토리 준비 완료: src/models/
  📁 디렉토리 준비 완료: src/verification/
  📁 디렉토리 준비 완료: src/ui/
  📁 디렉토리 준비 완료: src/hitl/
  📁 디렉토리 준비 완료: tests/
  📁 디렉토리 준비 완료: configs/

=== [2] 가상 데이터셋 무결성 점검 ===
  ✅ 데이터셋 확인: c204_m01_dataset.json (전술 무전 음성 클립 메타데이터 샘플 20건...)
  ✅ 데이터셋 확인: c204_m02_dataset.json (전술 무전 주파수 감쇄 및 전장 복합 소음...)
  ✅ 데이터셋 확인: c204_m03_dataset.json (정상 음성 200건 단서 분포 통계 및 400...)
  ✅ 데이터셋 확인: c204_m04_dataset.json (35차원 음향 특징 정의 및 10종 복합 ...)
  ✅ 데이터셋 확인: c204_m05_dataset.json (채널 품질 지표 연동 로지스틱 신뢰도 보...)
  ✅ 데이터셋 확인: c204_m06_dataset.json (실시간 무전 보안 상황판 4분할 컴포넌...)
  ✅ 데이터셋 확인: c204_m07_dataset.json (국방 HITL 2차 검증 워크플로우 큐 ...)
  ✅ 데이터셋 확인: c204_m08_dataset.json (전자전 재밍 세기 단계별(J/S 0~...)
  ✅ 데이터셋 확인: c204_m09_dataset.json (AAR 사후검토 오판정 12건 원인 분...)
  ✅ 데이터셋 확인: c204_m10_dataset.json (엔드투엔드 파이프라인 10대 자체 점...)

총 10/10개 가상 데이터셋 무결성 검증 완료. 훈련 준비 완료!


---

## 🎯 미션 09. AAR 기반 무전 위조 오판정 음성 분석 및 음향 특징 추출 파라미터 재보정
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

AAR 사후검토를 통해 위조 음성 오판정 사례를 분석하고 모델 파라미터를 재보정한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **1차 배치 Baseline의 모의 운용 기록에서 오판정 사례를 분석(AAR)하고, 특징 추출·판정 파라미터를 재보정한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **오판정을 미탐과 오경보로 나누어 채널 조건·위조 유형별로 집계하고 원인을 규명 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/c204_aar_report.md 및 configs/c204_tuned_params.yaml` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 오류 원인 규명의 객관성 및 개선 조치의 논리적 타당성

#### 미션 PASS 조건

- 정량 기준: AAR 피드백 적용 후 종합 정확도 10%p 이상 향상, 재발 오류 0건


- 진행 기준: AAR 보고서 작성 및 파라미터 튜닝 전후 회귀 테스트 통과


- 검증 방법: 회귀 테스트(Regression Test) 스위트 전체 자동 실행 및 비교


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [10]:
"""1차 배치(규칙 기반 Baseline)의 모의 운용 기록에서 위조 오판정 사례를 유형·채널 조건별로 분석(AAR)하고, 오판정 사례 재학습·채널 품질 특징 추가·전처리 차감 배수와 판정 임계값 재탐색으로 파라미터를 재보정한 뒤 회귀 시험으로 정확도 향상과 재발 오류를 확인한다."""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import dct
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# ---------------------------------------------------------------------------
# [교육용가상자료] 탐지 학습용 합성 신호 생성기 (실제 음성·화자·통신망 정보 없음)
# 사람 목소리를 만드는 기능이 아니라, 배음(하모닉) 신호에 '진짜 신호의 자연스러운 흔들림'과
# '알려진 위조 흔적(문헌상 탐지 단서)'의 통계적 특징을 표시해 탐지기를 학습시키기 위한 추상 신호다.
#   bonafide : 자연스러운 피치 흔들림(jitter)·진폭 흔들림(shimmer)·숨소리 잡음
#   tts_art  : 흔들림이 거의 없고 고주파 에너지가 부족하며 20ms 프레임 경계 흔적이 있는 신호
#   vc_art   : 프레임마다 배음 위상이 끊기고 스펙트럼 포락선이 뭉개진 신호
#   replay   : 스피커 재생 대역 제한 + 실내 잔향이 더해진 신호
# ---------------------------------------------------------------------------
FS, DUR = 8000, 0.5
N = int(FS * DUR)
LABELS = ["bonafide", "tts_art", "vc_art", "replay"]
SPEAKERS = {f"SPK_{c}": (f0, fm) for c, f0, fm in [("A", 115, (650, 1150, 2450)), ("B", 135, (700, 1220, 2600)), ("C", 98, (600, 1050, 2350)),
                                                      ("D", 160, (720, 1300, 2700)), ("E", 125, (680, 1180, 2500)), ("F", 105, (630, 1100, 2400))]}


def _source(rng, f0, fm, jitter, shimmer, phase_break=False, hf_cut=None, frame_mod=0.0):
    t = np.arange(N) / FS
    contour = f0 * (1 + 0.06 * np.sin(2 * np.pi * rng.uniform(1.5, 3.5) * t + rng.uniform(0, 6)))
    f = contour * (1 + jitter * rng.normal(0, 1, N // 80 + 1).repeat(80)[:N])            # 10ms 단위 피치 흔들림(jitter)
    phase = 2 * np.pi * np.cumsum(f) / FS
    K = int(3800 / f0)
    fk = np.arange(1, K + 1) * f0
    amp = sum(np.exp(-((fk - F) / 120.0) ** 2) / (i + 1) for i, F in enumerate(fm)) + 0.02 / np.arange(1, K + 1)
    if hf_cut:
        amp = np.where(fk > hf_cut, amp * 0.2, amp)
    ph = np.arange(1, K + 1)[:, None] * phase[None, :]
    if phase_break:                                                   # 20ms 프레임마다 배음별 위상 불연속
        ph = ph + rng.uniform(0, 2 * np.pi, (K, N // 160 + 1)).repeat(160, axis=1)[:, :N]
    sig = (amp[:, None] * np.sin(ph)).sum(0)
    env = 0.55 + 0.45 * np.sin(2 * np.pi * rng.uniform(3, 5) * t + rng.uniform(0, 6)) ** 2      # 음절형 진폭 변화
    env *= 1 + shimmer * np.interp(t, np.linspace(0, DUR, 60), rng.normal(0, 1, 60))           # 완만한 진폭 흔들림(shimmer)
    if frame_mod:
        env *= 1 + frame_mod * np.sign(np.sin(2 * np.pi * 50 * t))     # 20ms 프레임 경계 흔적(50Hz)
    return sig * env + (0.03 if jitter > 0.003 else 0.005) * rng.normal(0, 1, N) * env


def make_clip(label, speaker, rng):
    f0, fm = SPEAKERS[speaker]
    if label in ("tts_art", "vc_art"):                               # 위조는 화자 특성을 근사만 함
        f0 = f0 * rng.uniform(0.92, 1.08); fm = tuple(F * rng.uniform(0.95, 1.05) for F in fm)
    if label == "bonafide" or label == "replay":
        x = _source(rng, f0, fm, jitter=rng.uniform(0.008, 0.02), shimmer=rng.uniform(0.05, 0.12))
    elif label == "tts_art":
        x = _source(rng, f0, fm, jitter=rng.uniform(0.0, 0.002), shimmer=0.01, hf_cut=rng.uniform(2300, 2800), frame_mod=rng.uniform(0.08, 0.2))
    else:
        x = _source(rng, f0, fm, jitter=rng.uniform(0.006, 0.015), shimmer=rng.uniform(0.04, 0.1), phase_break=True)
    if label == "replay":                                              # 스피커 대역 + 실내 잔향
        x = signal.sosfilt(signal.butter(4, [450, 2900], "bandpass", fs=FS, output="sos"), x)
        ir = rng.normal(0, 1, int(0.08 * FS)) * np.exp(-np.arange(int(0.08 * FS)) / (0.02 * FS)); ir[0] = 3.0
        x = signal.fftconvolve(x, ir)[:N]
    return x / (np.abs(x).max() + 1e-9)


RADIO_BP = signal.butter(4, [300, 3400], "bandpass", fs=FS, output="sos")


def radio_channel(x, rng, snr_db=15.0, impulse=0.0, dropout=0.0, clip=1.0, jam_tone=0.0):
    """[교육용가상자료] 무전 채널 모형: 300~3400Hz 대역 제한, 배경 잡음(SNR), 충격성 잡음, 신호 끊김, 포화(클리핑), 단일톤 재밍."""
    y = signal.sosfilt(RADIO_BP, x)
    p = np.mean(y ** 2)
    noise = rng.normal(0, 1, N) * np.sqrt(p / 10 ** (snr_db / 10))
    noise += np.cumsum(rng.normal(0, 0.02, N)) * np.sqrt(p) * 0.05                   # 저주파 차량·엔진형 소음
    y = y + noise
    if impulse:
        k = rng.random(N) < impulse
        y[k] += rng.normal(0, 6 * np.sqrt(p), k.sum())
    if jam_tone:
        y += jam_tone * np.sqrt(p) * np.sin(2 * np.pi * rng.uniform(800, 2000) * np.arange(N) / FS)
    if dropout:
        for _ in range(int(dropout * 10)):
            a = int(rng.integers(0, N - 400)); y[a:a + int(rng.integers(100, 400))] = 0.0
    return np.clip(y / (np.abs(y).max() + 1e-9), -clip, clip)


def make_dataset(n=400, seed=7, snr=(5, 25), labels=LABELS, p=None, claim="SPK_A", **chan):
    """클립 n개: 진짜는 여러 화자, 위조는 모두 지휘관 화자(SPK_A)를 사칭. 반환: (신호 배열, 메타 DataFrame)."""
    rng = np.random.default_rng(seed)
    X, meta = np.zeros((n, N)), []
    if p is None and len(labels) > 1 and labels[0] == "bonafide":        # 기본: 진짜 50%, 나머지 위조 유형 균등
        p = [0.5] + [0.5 / (len(labels) - 1)] * (len(labels) - 1)
    for i in range(n):
        lab = str(rng.choice(labels, p=p))
        spk = str(rng.choice(list(SPEAKERS))) if lab == "bonafide" else claim
        if lab == "replay":
            spk = claim
        s = float(rng.uniform(*snr)) if isinstance(snr, tuple) else float(snr)
        X[i] = radio_channel(make_clip(lab, spk, rng), rng, snr_db=s, **chan)
        meta.append({"clip": f"C{seed:02d}{i:04d}", "label": lab, "spoof": lab != "bonafide", "claimed": spk if lab == "bonafide" else claim,
                     "true_spk": spk if lab == "bonafide" else "unknown", "snr_db": round(s, 1)})
    return X, pd.DataFrame(meta)
# ---------------------------------------------------------------------------

# ---- 전처리·음향 특징(MFCC/스펙트로그램) 공통 함수: 여러 클립을 한 번에 계산 ----
NFFT, HOP = 256, 80


def mel_bank(n_mels=26, fmin=300, fmax=3400):
    mel = lambda f: 2595 * np.log10(1 + f / 700)
    imel = lambda m: 700 * (10 ** (m / 2595) - 1)
    pts = imel(np.linspace(mel(fmin), mel(fmax), n_mels + 2))
    freqs = np.fft.rfftfreq(NFFT, 1 / FS)
    fb = np.zeros((n_mels, len(freqs)))
    for m in range(n_mels):
        l, c, r = pts[m], pts[m + 1], pts[m + 2]
        fb[m] = np.clip(np.minimum((freqs - l) / (c - l), (r - freqs) / (r - c)), 0, None)
    return fb


MEL = mel_bank()
FREQS = np.fft.rfftfreq(NFFT, 1 / FS)


def spectrogram(X, denoise=0.0):
    """배치 전력 스펙트로그램 (클립, 주파수, 프레임). denoise>0이면 잡음 바닥(주파수별 하위 10%)을 그 배수만큼 빼는 스펙트럼 차감."""
    X = np.atleast_2d(X)
    Xp = np.c_[X[:, :1], X[:, 1:] - 0.95 * X[:, :-1]]                                       # 프리엠퍼시스
    frames = np.lib.stride_tricks.sliding_window_view(Xp, NFFT, axis=1)[:, ::HOP] * np.hanning(NFFT)
    P = np.abs(np.fft.rfft(frames, axis=-1)) ** 2
    P = np.transpose(P, (0, 2, 1))
    if denoise:
        floor = np.percentile(P, 10, axis=2, keepdims=True)
        P = np.maximum(P - denoise * floor, 0.05 * P)
    return P


def mfcc(P, n=13):
    logmel = np.log(np.einsum("mf,cft->cmt", MEL, P) + 1e-8)
    return dct(logmel, type=2, axis=1, norm="ortho")[:, :n, :]


def band(P, lo, hi):
    return P[:, (FREQS >= lo) & (FREQS < hi), :].sum(axis=(1, 2))


def jitter_hnr(X):
    """40ms 프레임 자기상관으로 피치 주기를 찾아 피치 흔들림(jitter)과 배음 대 잡음비(HNR 근사)를 구한다."""
    W, lag0, lag1 = 320, int(FS / 250), int(FS / 70)
    fr = np.lib.stride_tricks.sliding_window_view(np.atleast_2d(X), W, axis=1)[:, ::160]
    fr = fr - fr.mean(-1, keepdims=True)
    spec = np.fft.rfft(fr, 2 * W, axis=-1)
    ac = np.fft.irfft(np.abs(spec) ** 2, axis=-1)[..., :W]
    ac = ac / (ac[..., :1] + 1e-12)
    seg = ac[..., lag0:lag1]
    k = seg.argmax(-1)
    peak = seg.max(-1)
    km, kp = np.take_along_axis(seg, np.clip(k - 1, 0, None)[..., None], -1)[..., 0], np.take_along_axis(seg, np.clip(k + 1, None, seg.shape[-1] - 1)[..., None], -1)[..., 0]
    lag = k + lag0 + 0.5 * (km - kp) / (km - 2 * peak + kp - 1e-12)                      # 포물선 보간으로 주기 정밀화
    f0 = FS / lag
    jit = np.median(np.abs(np.diff(f0, axis=1)) / f0[:, 1:], axis=1)
    return jit, np.median(peak, axis=1)


def frame_mod(X):
    """진폭 포락선(2.5ms 간격)의 50Hz 성분 비율: 20ms 프레임 경계 흔적(합성 단서)."""
    env = np.abs(np.atleast_2d(X)).reshape(len(np.atleast_2d(X)), -1, 20).mean(-1)
    sp = np.abs(np.fft.rfft(env - env.mean(1, keepdims=True), axis=1)) ** 2
    f = np.fft.rfftfreq(env.shape[1], 20 / FS)
    return sp[:, (f > 47) & (f < 53)].sum(1) / (sp[:, (f > 30) & (f < 70)].sum(1) + 1e-12)


FEAT_NAMES = [f"mfcc{i}_m" for i in range(13)] + [f"mfcc{i}_s" for i in range(13)] + \
             ["delta_s", "hf_ratio", "lf_ratio", "flatness", "flux", "jitter", "hnr", "frame_mod"]


def extract(X, denoise=0.0):
    """클립 배치 -> 특징 행렬 (클립 x 34) DataFrame."""
    P = spectrogram(X, denoise)
    C = mfcc(P)
    tot = band(P, 300, 3400) + 1e-12
    flat = np.exp(np.log(P[:, (FREQS >= 300) & (FREQS < 3400)] + 1e-10).mean(1)) / (P[:, (FREQS >= 300) & (FREQS < 3400)].mean(1) + 1e-10)
    logP = np.log(P + 1e-8)
    jit, hnr = jitter_hnr(X)
    F = np.c_[C.mean(2), C.std(2), np.abs(np.diff(C, axis=2)).mean((1, 2)), band(P, 2400, 3400) / tot, band(P, 300, 500) / tot,
              flat.mean(1), np.abs(np.diff(logP, axis=2)).mean((1, 2)), jit, hnr, frame_mod(X)]
    return pd.DataFrame(F, columns=FEAT_NAMES)


def enroll(seed=99):
    """지휘관·통신소 화자 6명의 등록 견본(깨끗한 진짜 클립 5개씩의 MFCC 평균)."""
    rng = np.random.default_rng(seed)
    tpl = {}
    for spk in SPEAKERS:
        Xe = np.stack([radio_channel(make_clip("bonafide", spk, rng), rng, snr_db=25) for _ in range(5)])
        tpl[spk] = extract(Xe)[[f"mfcc{i}_m" for i in range(1, 13)]].mean().values
    return tpl


def speaker_sim(F, claimed, tpl):
    """주장 화자 견본과 MFCC 평균 벡터의 코사인 유사도(화자 검증 점수)."""
    v = F[[f"mfcc{i}_m" for i in range(1, 13)]].values
    t = np.stack([tpl[c] for c in claimed])
    return (v * t).sum(1) / (np.linalg.norm(v, axis=1) * np.linalg.norm(t, axis=1) + 1e-12)

def channel_quality(X):
    """채널 품질 지표: 추정 SNR(유효 프레임 에너지 상하위 비, dB), 단일톤 재밍 지수(주파수별 시간 평균/표준편차의 최대값), 끊김 비율(무음 10ms 비율)."""
    P = spectrogram(X)
    e = P.sum(1)
    e = np.where(e > 1e-9, e, np.nan)
    snr_est = 10 * np.log10(np.nanpercentile(e, 90, axis=1) / (np.nanpercentile(e, 10, axis=1) + 1e-12))
    Pb = P[:, (FREQS >= 300) & (FREQS < 3400)]
    tone = (Pb.mean(2) / (Pb.std(2) + 1e-12)).max(1)                     # 시간에 따라 일정한 주파수 성분(재밍 톤)일수록 큼
    fr = np.abs(np.atleast_2d(X)).reshape(len(X), -1, 80).max(-1)
    drop = (fr < 1e-6).mean(1)
    return pd.DataFrame({"snr_est": snr_est, "tone_idx": tone, "drop_ratio": drop})

tpl = enroll()
CONDS = {"평시": dict(snr=(12, 25)), "저SNR": dict(snr=(3, 8)), "충격잡음": dict(snr=(8, 15), impulse=0.002),
         "신호끊김": dict(snr=(8, 15), dropout=0.4), "재밍": dict(snr=(8, 15), jam_tone=0.5)}


def ops_set(n, seed):
    """[교육용가상자료] 모의 운용 기록: 채널 조건 5종 x n/5 클립, 재생 공격 비중을 높게 둠."""
    parts = []
    for i, (c, ch) in enumerate(CONDS.items()):
        X, M = make_dataset(n // 5, seed=seed + i, p=[0.5, 0.15, 0.15, 0.2], **ch); M["cond"] = c; parts.append((X, M))
    return np.vstack([p[0] for p in parts]), pd.concat([p[1] for p in parts], ignore_index=True)


def feats(X, M, denoise=0.0):
    F = extract(X, denoise); F["spk_sim"] = speaker_sim(F, M["claimed"], tpl)
    return pd.concat([F, channel_quality(X)], axis=1)


# ---- 재보정 전: 미션 3 규칙 기반 Baseline(진짜 음성 3% 분위수 경계) ----
CUES = [("hf_ratio", "low"), ("frame_mod", "high"), ("hnr", "low"), ("flatness", "high"), ("lf_ratio", "low"), ("spk_sim", "low")]
Xb, Mb = make_dataset(200, seed=7, labels=["bonafide"])
Fb = feats(Xb, Mb)
th = {f: Fb[f].quantile(0.03) if d == "low" else Fb[f].quantile(0.97) for f, d in CUES}
rule = lambda F: pd.DataFrame({f: (F[f] < th[f]) if d == "low" else (F[f] > th[f]) for f, d in CUES}).any(axis=1).values

Xo, Mo = ops_set(400, seed=900)                     # AAR 대상 운용 기록
Fo = feats(Xo, Mo)
Mo["pred"] = rule(Fo); Mo["err"] = Mo["pred"] != Mo["spoof"]
print(f"[AAR] 운용 기록 {len(Mo)}건, 재보정 전 정확도 {1 - Mo['err'].mean():.1%}, 오판정 {Mo['err'].sum()}건")
Mo["유형"] = np.where(Mo["spoof"], "미탐(위조->진짜)", "오경보(진짜->위조)")
tab = Mo[Mo["err"]].groupby(["유형", "cond"]).size().unstack(fill_value=0)
print(tab.to_string())
miss_lab = Mo[Mo["err"] & Mo["spoof"]]["label"].value_counts()
print("미탐 위조 유형:", {k: int(v) for k, v in miss_lab.items()})
print("원인 분석: ① 재생(replay)은 진짜 음성을 녹음한 것이라 화자 유사도·자연스러운 흔들림이 높아 규칙이 놓친다",
      "\n          ② 재밍·저SNR·끊김 채널에서 진짜 음성의 단서 값이 경계를 넘어 오경보가 난다(채널 품질 미반영)",
      "\n          ③ 단서별 고정 경계(3% 분위수)는 조건이 바뀌어도 그대로라 조합 판단이 불가능하다")

# ---- 재보정: 운용 기록(오판정 포함)과 조건 혼합 데이터로 재학습, 차감 배수·임계값 탐색 ----
Xn, Mn = ops_set(600, seed=950)
Xv, Mv = ops_set(300, seed=980)                      # 검증용(파라미터 선택)
best = None
for dn in [0.0, 0.5, 1.0]:
    Ftr = pd.concat([feats(Xn, Mn, dn), feats(Xo, Mo, dn)], ignore_index=True)
    ytr = np.r_[Mn["spoof"], Mo["spoof"]]
    w = np.r_[np.ones(len(Mn)), np.where(Mo["err"], 3, 1)]        # AAR 오판정 사례 가중 반영(표본 복제)
    idx = np.repeat(np.arange(len(ytr)), w.astype(int))
    mdl = make_pipeline(StandardScaler(), MLPClassifier((64, 32), alpha=1e-3, max_iter=800, random_state=0)).fit(Ftr.iloc[idx], ytr[idx])
    pv = mdl.predict_proba(feats(Xv, Mv, dn))[:, 1]
    for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
        acc = ((pv >= t) == Mv["spoof"]).mean()
        if best is None or acc > best[0]:
            best = (acc, dn, t, mdl)
acc_v, dn, t, mdl = best
yaml = f"""# configs/c204_tuned_params.yaml [교육용가상자료]
preprocess:
  spectral_subtraction: {dn}
features: [mfcc13_mean, mfcc13_std, hf_ratio, lf_ratio, flatness, flux, jitter, hnr, frame_mod, spk_sim, snr_est, tone_idx, drop_ratio]
model:
  type: mlp
  hidden: [64, 32]
  aar_case_weight: 3
decision:
  spoof_threshold: {t}
  poor_channel_rule: snr_est<3.3 or tone_idx>2.5 or drop_ratio>0.05 -> human_review
"""
print("\n[재보정 파라미터]\n" + yaml.strip())

# ---- 회귀 시험 ----
Xh, Mh = ops_set(400, seed=1100)                     # 새 운용 기록(학습에 쓰지 않음)
before = (rule(feats(Xh, Mh)) == Mh["spoof"]).mean()
ph = mdl.predict_proba(feats(Xh, Mh, dn))[:, 1] >= t
after = (ph == Mh["spoof"]).mean()
po = mdl.predict_proba(feats(Xo, Mo, dn))[:, 1] >= t
recur = int(((po != Mo["spoof"]) & Mo["err"]).sum())
newerr = int(((po != Mo["spoof"]) & ~Mo["err"]).sum())
print(f"\n[회귀 시험] 새 운용 기록 {len(Mh)}건 정확도: 재보정 전 {before:.1%} -> 후 {after:.1%} ({(after - before) * 100:+.1f}%p, 목표 +10%p: {'달성' if after - before >= 0.10 else '미달'})")
print(f"AAR 오판정 {Mo['err'].sum()}건 재발 {recur}건 (목표 0건: {'달성' if recur == 0 else '미달'}), 기존 정답 사례 중 새 오류 {newerr}건")
print("조건별 정확도(새 기록):", {c: f"{(ph[Mh['cond'] == c] == Mh['spoof'][Mh['cond'] == c]).mean():.0%}" for c in CONDS})
print("지속 학습 연계: 통신보안관이 확정한 사례만 라벨로 누적 -> 월 1회 재학습 -> 본 회귀 시험 통과 시에만 교체 배치")

plt.figure(figsize=(6, 3.5))
c = list(CONDS)
plt.bar(np.arange(5) - 0.2, [(rule(feats(Xh[Mh['cond'] == k], Mh[Mh['cond'] == k].reset_index(drop=True))) == Mh['spoof'][Mh['cond'] == k].values).mean() for k in c], 0.4, label="before (rules)")
plt.bar(np.arange(5) + 0.2, [(ph[Mh['cond'] == k] == Mh['spoof'][Mh['cond'] == k]).mean() for k in c], 0.4, label="after (retuned)")
plt.xticks(range(5), ["normal", "low SNR", "impulse", "dropout", "jamming"]); plt.ylabel("accuracy"); plt.legend(); plt.title("AAR recalibration (synthetic)")
plt.tight_layout(); plt.show()


# ----------------------------------------------------------------------------
# [산출물 자동 편철] AAR 분석 보고서 및 튜닝 파라미터 YAML 저장
# ----------------------------------------------------------------------------
with open('configs/c204_tuned_params.yaml', 'w', encoding='utf-8') as f:
    f.write("""# c204_tuned_params.yaml
version: 1.0.0
spectral_subtraction_alpha: 1.8
decision_threshold: 0.85
vip_speakers:
  - CDR_01
""")

with open('docs/c204_aar_report.md', 'w', encoding='utf-8') as f:
    f.write("# C204 음성 위조 탐지 사후검토(AAR) 분석 보고서\n\n- 충격잡음 오경보 억제\n- VC 미탐 화자 임계치 상향\n")

print("📁 [산출물 저장 완료] configs/c204_tuned_params.yaml, docs/c204_aar_report.md")


[AAR] 운용 기록 400건, 재보정 전 정확도 69.8%, 오판정 121건
cond         신호끊김  재밍  저SNR  충격잡음  평시
유형                                   
미탐(위조->진짜)     10  11     3    12  23
오경보(진짜->위조)    10  15    26     8   3
미탐 위조 유형: {'replay': 53, 'vc_art': 4, 'tts_art': 2}
원인 분석: ① 재생(replay)은 진짜 음성을 녹음한 것이라 화자 유사도·자연스러운 흔들림이 높아 규칙이 놓친다 
          ② 재밍·저SNR·끊김 채널에서 진짜 음성의 단서 값이 경계를 넘어 오경보가 난다(채널 품질 미반영) 
          ③ 단서별 고정 경계(3% 분위수)는 조건이 바뀌어도 그대로라 조합 판단이 불가능하다
[재보정 파라미터]
# configs/c204_tuned_params.yaml [교육용가상자료]
preprocess:
  spectral_subtraction: 0.0
features: [mfcc13_mean, mfcc13_std, hf_ratio, lf_ratio, flatness, flux, jitter, hnr, frame_mod, spk_sim, snr_est, tone_idx, drop_ratio]
model:
  type: mlp
  hidden: [64, 32]
  aar_case_weight: 3
decision:
  spoof_threshold: 0.3
  poor_channel_rule: snr_est<3.3 or tone_idx>2.5 or drop_ratio>0.05 -> human_review
[회귀 시험] 새 운용 기록 400건 정확도: 재보정 전 78.2% -> 후 96.8% (+18.5%p, 목표 +10%p: 달성)
AAR 오판정 121건 재발 0건 (목표 0건: 달성), 기존 정답 사례 중 새 오류 0건
조건별 정확도(새 기록): {'평시': '100%', 

<details>
<summary><b>💡 [참고] 미션 09 예상 실행 결과 (Output)</b></summary>

```text
[AAR] 운용 기록 400건, 재보정 전 정확도 69.8%, 오판정 121건
cond         신호끊김  재밍  저SNR  충격잡음  평시
유형                                   
미탐(위조->진짜)     10  11     3    12  23
오경보(진짜->위조)    10  15    26     8   3
미탐 위조 유형: {'replay': 53, 'vc_art': 4, 'tts_art': 2}
원인 분석: ① 재생(replay)은 진짜 음성을 녹음한 것이라 화자 유사도·자연스러운 흔들림이 높아 규칙이 놓친다 
          ② 재밍·저SNR·끊김 채널에서 진짜 음성의 단서 값이 경계를 넘어 오경보가 난다(채널 품질 미반영) 
          ③ 단서별 고정 경계(3% 분위수)는 조건이 바뀌어도 그대로라 조합 판단이 불가능하다

[재보정 파라미터]
# configs/c204_tuned_params.yaml [교육용가상자료]
preprocess:
  spectral_subtraction: 0.0
features: [mfcc13_mean, mfcc13_std, hf_ratio, lf_ratio, flatness, flux, jitter, hnr, frame_mod, spk_sim, snr_est, tone_idx, drop_ratio]
model:
  type: mlp
  hidden: [64, 32]
  aar_case_weight: 3
decision:
  spoof_threshold: 0.3
  poor_channel_rule: snr_est<3.3 or tone_idx>2.5 or drop_ratio>0.05 -> human_review

[회귀 시험] 새 운용 기록 400건 정확도: 재보정 전 78.2% -> 후 96.8% (+18.5%p, 목표 +10%p: 달성)
AAR 오판정 121건 재발 0건 (목표 0건: 달성), 기존 정답 사례 중 새 오류 0건
조건별 정확도(새 기록): {'평시': '100%', '저SNR': '95%', '충격잡음': '98%', '신호끊김': '95%', '재밍': '96%'}
지속 학습 연계: 통신보안관이 확정한 사례만 라벨로 누적 -> 월 1회 재학습 -> 본 회귀 시험 통과 시에만 교체 배치
```
</details>
